# LSTM — Long Short-Term Memory (1997)

_Papers · Sequence & NLP_

**A recurrent cell with a protected memory and learned gates, so gradients survive across long time gaps.**

---

This notebook is a practice scaffold for the **LSTM — Long Short-Term Memory (1997)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Build one LSTM cell from scratch

An LSTM cell takes the current input `x`, the previous hidden state `h`, and the previous memory `c`, and produces new `h` and `c`. All four gates are computed in one matrix multiply, then sliced apart in PyTorch's packing order **i, f, g, o**.

The *input* gate decides how much of the new candidate to write, the *forget* gate decides how much old memory to keep, the *candidate* `g` is the proposed new content, and the *output* gate decides how much of the memory to read out. The memory update `c = f*c + i*g` is the Constant Error Carousel that lets gradients survive across long gaps.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

def my_lstm_cell(x, h, c, W_ih, W_hh, b_ih, b_hh, H):
    """One LSTM cell step from scratch. Gate order = PyTorch's packing: i, f, g, o."""
    # All four gate pre-activations in one shot: (B, 4H) stacked.
    gates = x @ W_ih.t() + b_ih + h @ W_hh.t() + b_hh

    # Slice the stacked pre-activations into the four gates.
    i_pre = gates[:, 0*H:1*H]
    f_pre = gates[:, 1*H:2*H]
    g_pre = gates[:, 2*H:3*H]
    o_pre = gates[:, 3*H:4*H]

    i = torch.sigmoid(i_pre)   # input gate  (write how much new?)
    f = torch.sigmoid(f_pre)   # forget gate (keep how much old?)  [1999/2000 addition]
    g = torch.tanh(g_pre)      # candidate   (the new content)
    o = torch.sigmoid(o_pre)   # output gate (release how much memory?)

    c_new = f * c + i * g           # cell-state update (the carousel + write)
    h_new = o * torch.tanh(c_new)   # gated, squashed read-out
    return h_new, c_new

### Step 2 — Check it against PyTorch's `nn.LSTMCell`

The cleanest test of a from-scratch implementation is to feed the **same weights** to both our cell and PyTorch's reference `nn.LSTMCell`, then confirm the outputs match to numerical precision. Note that `nn.LSTMCell` carries two bias vectors (`bias_ih` and `bias_hh`), so we pass both.

In [ ]:
# THE ORACLE: my cell must equal nn.LSTMCell on the SAME weights.
I, H, B = 3, 4, 2
cell = nn.LSTMCell(I, H)   # PyTorch's reference cell

x = torch.randn(B, I)
h0 = torch.zeros(B, H)     # PyTorch defaults states to zero
c0 = torch.zeros(B, H)

# Reference outputs from PyTorch.
h_ref, c_ref = cell(x, (h0, c0))

# Our outputs, given the SAME weights and biases.
h_mine, c_mine = my_lstm_cell(
    x, h0, c0,
    cell.weight_ih.detach(), cell.weight_hh.detach(),   # (4H, I) and (4H, H)
    cell.bias_ih.detach(),   cell.bias_hh.detach(),     # nn.LSTMCell adds BOTH biases
    H)

print("h allclose :", torch.allclose(h_mine, h_ref, atol=1e-6))   # True
print("c allclose :", torch.allclose(c_mine, c_ref, atol=1e-6))   # True
print("max |dh|   :", (h_mine - h_ref).abs().max().item())        # ~0

### Step 3 — Recompute the worked example by hand

To make the gate arithmetic concrete, we take a single unit (`H=1`) with fixed gate pre-activations and a previous memory `c_prev = 2`. We apply sigmoid to the input/forget/output gates and tanh to the candidate, then the same `c = f*c + i*g` and `h = o*tanh(c)` updates — this time with plain Python `math`, so you can trace every number.

In [ ]:
# Recompute the WORKED EXAMPLE: H=1, gate nets z_i=0.5, z_f=1.0, z_g=0.8, z_o=0.3, c_prev=2.
import math

def sig(z):
    return 1 / (1 + math.exp(-z))

zi, zf, zg, zo, c_prev = 0.5, 1.0, 0.8, 0.3, 2.0

i = sig(zi)
f = sig(zf)
g = math.tanh(zg)
o = sig(zo)

c_t = f * c_prev + i * g
h_t = o * math.tanh(c_t)

print("i,f,g,o    :", [round(v, 6) for v in (i, f, g, o)])   # [0.622459, 0.731059, 0.664037, 0.574443]
print("c_t        :", round(c_t, 6))                         # 1.875453
print("h_t        :", round(h_t, 6))                         # 0.548068

### Step 4 — Train the cell on a long-gap memory task

Now we put the cell to work. Each sequence shows a +1/-1 **cue** at step 0, then 11 steps of pure noise, and asks the model to recall the cue at the end. We carry `h` and `c` across all 12 steps, read out from the final hidden state, and train with `Adam`. A trained LSTM reaches ~100% accuracy: it learns to latch the cue into memory and hold it across the gap.

Then we **ablate** the memory carry by zeroing the recurrent weights (`W_hh`, `b_hh`) and re-test without retraining. Accuracy collapses to chance (~0.5), proving that the recurrent carousel — not the input mapping — is what bridged the gap.

In [ ]:
# Use the cell on a 12-step LONG-GAP memory task.
T, Hd = 12, 16

def make_batch(n, g):
    # Cue at t=0, noise after, recall target is the cue.
    X = torch.zeros(n, T, 2)
    b = torch.randint(0, 2, (n,), generator=g).float()
    X[:, 0, 0] = b * 2 - 1                            # +1/-1 cue on feature 0 at step 0
    X[:, :, 1] = torch.randn(n, T, generator=g) * 0.5  # noise on feature 1 everywhere
    X[:, 0, 1] = 0.0
    return X, b

def run(cell, head, X):
    n = X.size(0)
    h = torch.zeros(n, Hd)
    c = torch.zeros(n, Hd)
    for t in range(T):
        h, c = cell(X[:, t, :], (h, c))   # carry h,c across all 12 steps
    return head(h).squeeze(1)

torch.manual_seed(0)
g = torch.Generator().manual_seed(100)

mem = nn.LSTMCell(2, Hd)
head = nn.Linear(Hd, 1)
opt = torch.optim.Adam(list(mem.parameters()) + list(head.parameters()), lr=0.03)
lossf = nn.BCEWithLogitsLoss()

ge = torch.Generator().manual_seed(999)
Xt, bt = make_batch(1000, ge)

def acc():
    preds = (torch.sigmoid(run(mem, head, Xt)) > 0.5).float()
    return (preds == bt).float().mean().item()

for step in range(601):
    X, b = make_batch(64, g)
    opt.zero_grad()
    loss = lossf(run(mem, head, X), b)
    loss.backward()
    opt.step()

print("trained gap-task accuracy:", round(acc(), 3))   # ~1.0 — it remembers across the gap

# ABLATION: break the memory carry, re-test (no retrain).
abl = nn.LSTMCell(2, Hd)
abl.load_state_dict(mem.state_dict())
with torch.no_grad():
    abl.weight_hh.zero_()   # no recurrence -> no carry
    abl.bias_hh.zero_()

with torch.no_grad():
    abl_preds = (torch.sigmoid(run(abl, head, Xt)) > 0.5).float()
    abl_acc = (abl_preds == bt).float().mean().item()

print("ablated (no recurrence)  :", round(abl_acc, 3))   # ~0.50 — chance; memory is what mattered

## Visualize the data & results

_Show an LSTM cell a cue at step 0, then 11 steps of noise, and ask it to recall the cue at the end. Does it learn to remember across the gap — and does breaking the memory carry destroy that?_

### Step 5 — Rebuild the task and watch accuracy climb

The visualization re-creates the same long-gap setup (a self-contained re-import keeps this cell runnable on its own). This time we **print accuracy every 50 steps** so you can watch it start at chance (~0.5) and jump to 1.0 once the cell learns to latch the cue into memory.

In [ ]:
import torch
import torch.nn as nn

T, Hd = 12, 16

def make_batch(n, g):
    X = torch.zeros(n, T, 2)
    b = torch.randint(0, 2, (n,), generator=g).float()
    X[:, 0, 0] = b * 2 - 1
    X[:, :, 1] = torch.randn(n, T, generator=g) * 0.5
    X[:, 0, 1] = 0.0
    return X, b

def run(cell, head, X):
    n = X.size(0)
    h = torch.zeros(n, Hd)
    c = torch.zeros(n, Hd)
    for t in range(T):
        h, c = cell(X[:, t, :], (h, c))
    return head(h).squeeze(1)

torch.manual_seed(0)
g = torch.Generator().manual_seed(100)

mem = nn.LSTMCell(2, Hd)
head = nn.Linear(Hd, 1)
opt = torch.optim.Adam(list(mem.parameters()) + list(head.parameters()), lr=0.03)
lossf = nn.BCEWithLogitsLoss()

ge = torch.Generator().manual_seed(999)
Xt, bt = make_batch(1000, ge)

def acc():
    preds = (torch.sigmoid(run(mem, head, Xt)) > 0.5).float()
    return round((preds == bt).float().mean().item(), 3)

for step in range(601):
    if step % 50 == 0:
        print("step", step, "acc", acc())   # 0.505 ... 0.495 -> 1.0 by step 150
    X, b = make_batch(64, g)
    opt.zero_grad()
    loss = lossf(run(mem, head, X), b)
    loss.backward()
    opt.step()

print("final acc", acc())   # 1.0

### Step 6 — Ablate the recurrence and confirm the carry mattered

With the trained model in hand, we copy its weights, zero the recurrent path (`W_hh`, `b_hh`), and re-test. Because the cue appears only at step 0 and the readout is at step 11, the model can only succeed by carrying memory across the gap. Removing the recurrence drops accuracy back to chance — direct evidence that the Constant Error Carousel is what made the 100% possible.

In [ ]:
abl = nn.LSTMCell(2, Hd)
abl.load_state_dict(mem.state_dict())
with torch.no_grad():
    abl.weight_hh.zero_()
    abl.bias_hh.zero_()

with torch.no_grad():
    abl_preds = (torch.sigmoid(run(abl, head, Xt)) > 0.5).float()
    abl_acc = (abl_preds == bt).float().mean().item()

print("ablation acc", round(abl_acc, 3))   # ~0.495

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Compute one LSTM cell step ($H=1$) with $x_t$ giving gate nets $z_i=0,\ z_f=2,\ z_g=0,\ z_o=0$, previous memory $c_{t-1}=1$, and $h_{t-1}=0$. Find $c_t$ and $h_t$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Gates: $i=\sigma(0)=0.5$, $f=\sigma(2)=0.880797$, $g=\tanh(0)=0$, $o=\sigma(0)=0.5$. — _Sigmoid on $i,f,o$; $\tanh$ on the candidate $g$._
- Memory: $c_t = f\cdot c_{t-1} + i\cdot g = 0.880797\cdot1 + 0.5\cdot0 = 0.880797$. — _Keep $f$ of old memory; add $i$ of the (here zero) candidate._
- Output: $h_t = o\cdot\tanh(c_t) = 0.5\cdot\tanh(0.880797) = 0.5\cdot0.706727 = 0.353363$. — _Output gate releases a squashed view of memory._

**Answer:** $c_t \approx 0.880797$, $h_t \approx 0.353363$. With $g=0$ nothing new is written, so the cell mostly carries its old memory forward (forget gate $0.88$) — a small demo of the carousel holding a value.

</details>

**Problem 2.** Why does setting the forget gate near $1$ and the input gate near $0$ let a value survive a long gap, while a plain RNN's hidden state does not?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- With $f\approx1,\ i\approx0$: $c_t = f\odot c_{t-1}+i\odot g \approx c_{t-1}$. — _The memory copies itself forward almost unchanged._
- Backward, $\partial c_t/\partial c_{t-1}\approx f\approx 1$, so across $k$ steps the factor is $\prod f \approx 1$. — _Near-identity carry = no vanishing._
- A plain RNN's carry factor is $\approx W\cdot\tanh' < 1$, whose $k$-fold product decays to $0$. — _Repeated multiplication by a sub-unit number vanishes exponentially._

**Answer:** The LSTM's memory has a (near) weight-$1$ self-loop — the Constant Error Carousel — so both the value and its gradient pass through ~unchanged. A plain RNN multiplies by a sub-unit factor every step, so the signal connecting distant steps shrinks toward zero. That is exactly the vanishing-gradient problem the 1997 paper solved.

</details>

**Problem 3.** Ablation: in the CODEVIZ long-gap task the trained LSTM reaches ~100% accuracy. We then zero the recurrent weights ($W_{hh}\!=\!0,\ b_{hh}\!=\!0$) and re-test without retraining. Predict the accuracy and explain.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Zeroing $W_{hh},b_{hh}$ removes the $h_{t-1}$ contribution to every gate. — _The cell can no longer condition on its own past output._
- Each gate now depends only on the current $x_t$; the cell cannot route the $t=0$ cue to step $11$. — _The memory carry that bridged the gap is broken._
- The last step is pure noise, so the readout has no information about the cue. — _Cue lives only at $t=0$; without recurrence it is unreachable at $t=11$._

**Answer:** Accuracy collapses to chance (~0.50; our run gave 0.495). The cue is shown only at step 0 and the readout is at step 11, so the model can only succeed by carrying memory across the gap. Destroying the recurrent path removes that carry — the Constant Error Carousel is what made the 100% possible, and the ablation proves it.

</details>